# Synthetic SMS Spam Dataset Generation using DistilGPT2


In [1]:
# Install transformers
!pip install transformers

# Import necessary libraries
import random
import torch
from transformers import pipeline


Load distilgpt2 Generator (Lightweight GPT-2)

In [2]:
# Use GPU if available
device = 0 if torch.cuda.is_available() else -1

# Load text generation pipeline with distilgpt2
generator = pipeline("text-generation", model="distilgpt2", device=device)

print("Model loaded on", "GPU" if device == 0 else "CPU")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu


Model loaded on CPU


Define Prompts for Spam and Ham

In [3]:
# Spam prompts that resemble realistic spam messages
spam_prompts = [
    "Click on the link below to win",
    "Store special sale offer item for 2$",
    "You won a prize in our lucky draw",
    "Exclusive offer just for you",
    "Claim your free gift now"
]

# Ham prompts that resemble real conversations
ham_prompts = [
    "Hi bro how are you doing today",
    "See you later, thanks for the help with homework",
    "Will you come online to play games",
    "Let's meet up for lunch",
    "Hey, what are your weekend plans?"
]



Batch Generation Function (Faster)

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Load the tokenizer and model for DistilGPT-2
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Create a text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


Device set to use cpu


In [5]:
# ⚡ Generate multiple sequences at once for faster output
def generate_batch(prompts, label, num_return_sequences=3, max_new_tokens=40):
    results = []

    outputs = generator(
        prompts,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False  # This ensures output only includes the generated part
    )

    # Flatten output list
    flat_outputs = [item for sublist in outputs for item in sublist]

    for out in flat_outputs:
        text = out['generated_text'].strip().replace('\t', ' ').replace('\n', ' ')
        results.append((label, text))

    return results



Generate 2000 Spam and 2000 Ham Messages

In [ ]:
synthetic_data = []

# Set target count
spam_target = 2000
ham_target = 2000

# Generate spam messages
print("Generating spam messages...")
while len([x for x in synthetic_data if x[0] == 'spam']) < spam_target:
    prompts = random.choices(spam_prompts, k=3)
    batch = generate_batch(prompts, "spam", num_return_sequences=4)
    synthetic_data.extend(batch)

# Generate ham messages
print("Generating ham messages...")
while len([x for x in synthetic_data if x[0] == 'ham']) < ham_target:
    prompts = random.choices(ham_prompts, k=3)
    batch = generate_batch(prompts, "ham", num_return_sequences=4)
    synthetic_data.extend(batch)

# Truncate to exact size
spam_data = [x for x in synthetic_data if x[0] == 'spam'][:spam_target]
ham_data = [x for x in synthetic_data if x[0] == 'ham'][:ham_target]
synthetic_data = spam_data + ham_data

# Shuffle the dataset
random.shuffle(synthetic_data)

print(f"= Done! Generated {len(spam_data)} spam and {len(ham_data)} ham messages.")


Generating spam messages...
Generating ham messages...


Save as Tab-Separated CSV (SMSSpamCollection Format)

In [7]:
import csv

# Save file just like original SMSSpamCollection (no header, tab-separated)
with open("DistilGPT2_Synthetic_SMSSpamCollection.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter='\t', quoting=csv.QUOTE_MINIMAL)
    for label, message in synthetic_data:
        writer.writerow([label, message.strip()])

print("📁 Saved to 'DistilGPT2_Synthetic_SMSSpamCollection.csv'")


📁 Saved to 'DistilGPT2_Synthetic_SMSSpamCollection.csv'


In [8]:
import pandas as pd

df = pd.read_csv("DistilGPT2_Synthetic_SMSSpamCollection.csv", sep='\t', names=["label", "message"])
df.sample(10)  # Show 10 random samples


,label,message
1851,spam,.” The last word from the company is that i...
1373,ham,".‹ ""Alright, now I wanna eat. Now I want to p..."
907,spam,the prize!
1411,spam,. This one will give us our last entry. The pr...
3369,ham,", or are you having any difficulties with how ..."
43,ham,", play games, write or play games?"
822,ham,and start talking about the future of our coun...
2571,ham,?” she asked. I couldn't be more grateful for ...
3079,ham,. The guys were in the middle of the street in...
378,spam,". I am just a one year old. So, this may sou..."
